# Task 2: Custom Byte-Pair Encoding (BPE) Tokenizer & Autoregressive Causal Language Model

## Objective

Build a custom Byte Pair Encoding (BPE) tokenizer from scratch and implement a simple autoregressive causal language model training pipeline using PyTorch tensor operations.

### Tech Stack

- Python
- NumPy
- PyTorch

In [1]:
import re
from collections import Counter

import numpy as np
import torch

torch.manual_seed(42)
np.random.seed(42)

## Step 1: Prepare Training Corpus

In [2]:
corpus = [
    "Artificial Intelligence is transforming the world",
    "Machine Learning is a subset of Artificial Intelligence",
    "Deep Learning powers modern AI systems",
    "Transformers use self attention mechanisms",
    "Generative AI creates text images and code"
]

print(corpus)

['Artificial Intelligence is transforming the world', 'Machine Learning is a subset of Artificial Intelligence', 'Deep Learning powers modern AI systems', 'Transformers use self attention mechanisms', 'Generative AI creates text images and code']


## Step 2: Text Preprocessing

In [3]:
def preprocess(text):

    text = text.lower()

    text = re.sub(r'[^a-z ]','',text)

    return text

corpus = [preprocess(sentence) for sentence in corpus]

corpus

['artificial intelligence is transforming the world',
 'machine learning is a subset of artificial intelligence',
 'deep learning powers modern ai systems',
 'transformers use self attention mechanisms',
 'generative ai creates text images and code']

## Step 3: Initialize Character Vocabulary

In [4]:
vocab = {}

for sentence in corpus:

    for word in sentence.split():

        vocab[" ".join(word) + " </w>"] = vocab.get(" ".join(word) + " </w>",0)+1

vocab

{'a r t i f i c i a l </w>': 2,
 'i n t e l l i g e n c e </w>': 2,
 'i s </w>': 2,
 't r a n s f o r m i n g </w>': 1,
 't h e </w>': 1,
 'w o r l d </w>': 1,
 'm a c h i n e </w>': 1,
 'l e a r n i n g </w>': 2,
 'a </w>': 1,
 's u b s e t </w>': 1,
 'o f </w>': 1,
 'd e e p </w>': 1,
 'p o w e r s </w>': 1,
 'm o d e r n </w>': 1,
 'a i </w>': 2,
 's y s t e m s </w>': 1,
 't r a n s f o r m e r s </w>': 1,
 'u s e </w>': 1,
 's e l f </w>': 1,
 'a t t e n t i o n </w>': 1,
 'm e c h a n i s m s </w>': 1,
 'g e n e r a t i v e </w>': 1,
 'c r e a t e s </w>': 1,
 't e x t </w>': 1,
 'i m a g e s </w>': 1,
 'a n d </w>': 1,
 'c o d e </w>': 1}

## Step 4: Count Symbol Pairs

In [5]:
def get_stats(vocab):

    pairs = Counter()

    for word,freq in vocab.items():

        symbols = word.split()

        for i in range(len(symbols)-1):

            pairs[(symbols[i],symbols[i+1])] += freq

    return pairs

pairs = get_stats(vocab)

pairs.most_common(10)

[(('s', '</w>'), 8),
 (('e', '</w>'), 7),
 (('i', 'n'), 6),
 (('t', 'e'), 6),
 (('a', 'r'), 4),
 (('t', 'i'), 4),
 (('g', 'e'), 4),
 (('e', 'n'), 4),
 (('a', 'n'), 4),
 (('e', 'r'), 4)]

## Step 5: Merge the Most Frequent Pair

In [6]:
def merge_vocab(pair,vocab):

    new_vocab = {}

    bigram = " ".join(pair)

    replacement = "".join(pair)

    for word in vocab:

        new_word = word.replace(bigram,replacement)

        new_vocab[new_word]=vocab[word]

    return new_vocab

best = max(pairs,key=pairs.get)

print("Best Pair :",best)

vocab = merge_vocab(best,vocab)

vocab

Best Pair : ('s', '</w>')


{'a r t i f i c i a l </w>': 2,
 'i n t e l l i g e n c e </w>': 2,
 'i s</w>': 2,
 't r a n s f o r m i n g </w>': 1,
 't h e </w>': 1,
 'w o r l d </w>': 1,
 'm a c h i n e </w>': 1,
 'l e a r n i n g </w>': 2,
 'a </w>': 1,
 's u b s e t </w>': 1,
 'o f </w>': 1,
 'd e e p </w>': 1,
 'p o w e r s</w>': 1,
 'm o d e r n </w>': 1,
 'a i </w>': 2,
 's y s t e m s</w>': 1,
 't r a n s f o r m e r s</w>': 1,
 'u s e </w>': 1,
 's e l f </w>': 1,
 'a t t e n t i o n </w>': 1,
 'm e c h a n i s m s</w>': 1,
 'g e n e r a t i v e </w>': 1,
 'c r e a t e s</w>': 1,
 't e x t </w>': 1,
 'i m a g e s</w>': 1,
 'a n d </w>': 1,
 'c o d e </w>': 1}

## Step 6: Repeat BPE Merging

In [7]:
num_merges = 10

for i in range(num_merges):

    pairs = get_stats(vocab)

    if not pairs:
        break

    best = max(pairs,key=pairs.get)

    vocab = merge_vocab(best,vocab)

print(vocab)

{'ar ti f i c i a l </w>': 2, 'in te l l i gen c e</w>': 2, 'i s</w>': 2, 't r an s f or m in g </w>': 1, 't h e</w>': 1, 'w or l d </w>': 1, 'm a c h in e</w>': 1, 'l e ar n in g </w>': 2, 'a </w>': 1, 's u b s e t </w>': 1, 'o f </w>': 1, 'd e e p </w>': 1, 'p o w er s</w>': 1, 'm o d er n </w>': 1, 'a i </w>': 2, 's y s te m s</w>': 1, 't r an s f or m er s</w>': 1, 'u s e</w>': 1, 's e l f </w>': 1, 'a t te n ti o n </w>': 1, 'm e c h an i s m s</w>': 1, 'gen er a ti v e</w>': 1, 'c r e a te s</w>': 1, 'te x t </w>': 1, 'i m a ge s</w>': 1, 'an d </w>': 1, 'c o d e</w>': 1}


## Step 7: Build Vocabulary

In [8]:
tokens = set()

for word in vocab:

    for token in word.split():

        tokens.add(token)

tokens = sorted(tokens)

print(tokens)

print("\nVocabulary Size :",len(tokens))

['</w>', 'a', 'an', 'ar', 'b', 'c', 'd', 'e', 'e</w>', 'er', 'f', 'g', 'ge', 'gen', 'h', 'i', 'in', 'l', 'm', 'n', 'o', 'or', 'p', 'r', 's', 's</w>', 't', 'te', 'ti', 'u', 'v', 'w', 'x', 'y']

Vocabulary Size : 34


## Step 8: Token Encoding

In [9]:
token_to_id = {token:i for i,token in enumerate(tokens)}

id_to_token = {i:token for token,i in token_to_id.items()}

token_to_id

{'</w>': 0,
 'a': 1,
 'an': 2,
 'ar': 3,
 'b': 4,
 'c': 5,
 'd': 6,
 'e': 7,
 'e</w>': 8,
 'er': 9,
 'f': 10,
 'g': 11,
 'ge': 12,
 'gen': 13,
 'h': 14,
 'i': 15,
 'in': 16,
 'l': 17,
 'm': 18,
 'n': 19,
 'o': 20,
 'or': 21,
 'p': 22,
 'r': 23,
 's': 24,
 's</w>': 25,
 't': 26,
 'te': 27,
 'ti': 28,
 'u': 29,
 'v': 30,
 'w': 31,
 'x': 32,
 'y': 33}

## Step 9: Convert Tokens into Tensor

In [10]:
sample = list(tokens)

input_tensor = torch.tensor(
    [token_to_id[token] for token in sample],
    dtype=torch.long
)

input_tensor

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33])

## Step 10: Create Causal Mask

In [11]:
length = len(input_tensor)

mask = torch.triu(
    torch.ones(length,length),
    diagonal=1
)

mask = mask.bool()

mask

tensor([[False,  True,  True,  ...,  True,  True,  True],
        [False, False,  True,  ...,  True,  True,  True],
        [False, False, False,  ...,  True,  True,  True],
        ...,
        [False, False, False,  ..., False,  True,  True],
        [False, False, False,  ..., False, False,  True],
        [False, False, False,  ..., False, False, False]])

## Step 11: Embedding Layer

In [12]:
embedding = torch.nn.Embedding(len(tokens),32)

embedded = embedding(input_tensor)

embedded.shape

torch.Size([34, 32])

## Step 12: Simple Autoregressive Prediction

In [13]:
linear = torch.nn.Linear(32,len(tokens))

logits = linear(embedded)

print(logits.shape)

torch.Size([34, 34])


## Step 13: Predict Next Token

In [14]:
prediction = torch.argmax(logits,dim=-1)

prediction

tensor([22,  0, 25, 31, 33, 24,  0,  3,  5, 21, 26, 18, 22, 25, 18, 30, 26, 20,
        11, 24,  1, 23, 20, 20,  5, 22, 22, 24, 11,  1, 21, 23, 31,  3])

## Step 14: Decode Tokens

In [15]:
decoded = [id_to_token[i.item()] for i in prediction]

decoded

['p',
 '</w>',
 's</w>',
 'w',
 'y',
 's',
 '</w>',
 'ar',
 'c',
 'or',
 't',
 'm',
 'p',
 's</w>',
 'm',
 'v',
 't',
 'o',
 'g',
 's',
 'a',
 'r',
 'o',
 'o',
 'c',
 'p',
 'p',
 's',
 'g',
 'a',
 'or',
 'r',
 'w',
 'ar']

# Conclusion

In this notebook we:

- Built a custom BPE tokenizer
- Created a vocabulary
- Encoded tokens
- Built a causal mask
- Created token embeddings
- Generated logits
- Predicted the next token using a simple autoregressive pipeline

This demonstrates the fundamental concepts behind subword tokenization and causal language modeling.